# Topical Lectures, Controls
## Flight data, extra: what the files hold

Andreas Freise, 01.11.2026

This notebook needs no drone. It is for the members of the group who have the flight files, a copy of the `data/` folder of the drone laptop, and time to look at them. Every flight saves five files: the loop records, position, velocity, target and feedback at 50 Hz; the state of the drone, position, height, roll, pitch and yaw; the sensors of the flow deck, range, optical flow and velocity; the power, battery voltage and the four motors; and the landing record. The drone notebooks look at two columns of the first file. Here the rest is drawn, what the drone did while our loop was flying it. The task is presentation: find the view that explains a flight, polish it, and make it a figure for the summary.


In [ ]:
# Importing packages
import numpy as np
import matplotlib.pyplot as plt
import module
import crazyflie

%matplotlib inline

## Loading

`crazyflie.load_runs` loads every saved run of a folder, oldest first, and the reference recordings serve while the files of your group are not there yet. The cell lists the runs by name; `RUN` picks one, `None` takes the last real flight; model flights have only the loop records, so the views of height, attitude, battery and flow need a real one. The helper gives every record its time since the step of the run, so that all files share one time axis.


In [ ]:
DATA = 'data'
runs = [r for r in crazyflie.load_runs(DATA) if r['t_step'] is not None]
if not runs:
    DATA = 'data/reference'
    runs = [r for r in crazyflie.load_runs(DATA) if r['t_step'] is not None]
    print('no flights in data/ yet, using the recorded reference flights')
for r in runs:
    print(f"{r['label']:30s}  drone {str(r['drone']):6s}  axes {r['axes']:2s}  {len(r['pid']['t_s'])} loop samples")

RUN = None                      # a label from the list, None for the last real flight
real = [x for x in runs if x['drone'] != 'model']        # model flights have only the loop records
r = real[-1] if RUN is None else next(x for x in runs if x['label'] == RUN)

def t_since_step(r, group):
    """Time axis of one record group of a run, in seconds since the step of the target."""
    if group not in r:
        raise KeyError(f"{r['label']} has no {group} records: a model flight; pick a real flight with RUN")
    pid = r['pid']
    t0_ms = pid['t_ms'][0] - 1000.0 * pid['t_s'][0]        # the clock of the drone at loop time zero
    return (r[group]['t_ms'] - t0_ms) / 1000.0 - r['t_step']

print('showing', r['label'])


## Height and attitude

The height estimate of the drone against the range sensor of the flow deck, and roll and pitch against time, all from the state and the sensor files. The step is a pitch movement: how large is it, how long does it last, and does the height hold during it? A height that departs from the range sensor means the estimator has lost the floor.


In [ ]:
ts, tq = t_since_step(r, 'state'), t_since_step(r, 'sensor')
st, se = r['state'], r['sensor']
fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(ts, st['stateEstimate.z'], label='height estimate')
ax[0].plot(tq, se['range.zrange'] / 1000.0, alpha=0.7, label='range sensor')
ax[0].set_ylabel('height [m]')
ax[0].legend(loc='lower right')
ax[0].grid()
ax[1].plot(ts, st['stabilizer.roll'], label='roll')
ax[1].plot(ts, st['stabilizer.pitch'], label='pitch')
ax[1].axvline(0, color='k', lw=0.5, ls='--')
ax[1].set_xlim(-5, 9)
ax[1].set_xlabel('time since step [s]')
ax[1].set_ylabel('angle [deg]')
ax[1].legend(loc='upper right')
ax[1].grid()
plt.tight_layout()


## Battery and motors

The battery voltage under load and the four motor outputs, from the power file. The voltage sags when the motors work, and the spread between the motors shows the tilt and the trim of the drone. The same gains at the beginning and at the end of the afternoon fly on different batteries; this is where to look when two such flights differ.


In [ ]:
tp, pw = t_since_step(r, 'power'), r['power']
fig, ax = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
ax[0].plot(tp, pw['pm.vbat'])
ax[0].set_ylabel('battery [V]')
ax[0].grid()
for m in ('m1', 'm2', 'm3', 'm4'):
    ax[1].plot(tp, pw[f'motor.{m}'] / 65535.0 * 100.0, label=m)
ax[1].axvline(0, color='k', lw=0.5, ls='--')
ax[1].set_xlim(-5, 9)
ax[1].set_xlabel('time since step [s]')
ax[1].set_ylabel('motor output [%]')
ax[1].legend(loc='upper right')
ax[1].grid()
plt.tight_layout()


## The flow sensor

The optical flow counts of the flow deck while the drone moves. The position estimate is built from them; zero counts while the drone moves means the sensor lost the floor, and the estimate drifts from then on. The range sensor is drawn with them, since the flow is scaled by the height.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tq, se['motion.deltaX'], label='flow x [counts]')
ax.plot(tq, se['motion.deltaY'], label='flow y [counts]')
ax.axvline(0, color='k', lw=0.5, ls='--')
ax.set_xlim(-5, 9)
ax.set_xlabel('time since step [s]')
ax.set_ylabel('optical flow [counts per sample]')
ax.legend(loc='upper right')
ax.grid()


## The path in the plane, with time

The position in the plane, coloured by the time since the step, with the start and the target marked. For a step on one axis it is a line that shows how straight the drone flew; for a step on both axes it shows whether the two axes were equally fast.


In [ ]:
pid = r['pid']
t = pid['t_s'] - r['t_step']
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(pid['x'] - pid['target_x'][0], pid['y'] - pid['target_y'][0], c=t, s=6, cmap='viridis')
ax.plot(0, 0, 'k+', ms=12, mew=2)
ax.plot(pid['target_x'][-1] - pid['target_x'][0], pid['target_y'][-1] - pid['target_y'][0], 'kx', ms=12, mew=2)
plt.colorbar(sc, ax=ax, label='time since step [s]')
ax.set_xlabel('x from start [m]')
ax.set_ylabel('y from start [m]')
ax.set_aspect('equal')
ax.grid()


## Feedback and velocity

What our loop asked for and what the drone did, against the time since the step, from the loop records. The lag between the two curves is the response time of the velocity loop; a flat top on the feedback is the clamp at `v_max`.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for a in r['axes']:
    line, = ax.plot(t, pid[f'fb_{a}'], label=f'{a} feedback')
    ax.plot(t, pid[f'v{a}'], color=line.get_color(), alpha=0.5, label=f'{a} measured velocity')
ax.axvline(0, color='k', lw=0.5, ls='--')
ax.set_xlim(-1, 8)
ax.set_xlabel('time since step [s]')
ax.set_ylabel('velocity [m/s]')
ax.legend(loc='upper right')
ax.grid()


## Tasks

1. Pick the flight of your group that tells the most and make one figure of two panels for the summary: the step response and the view from this notebook that explains it. Axis labels with units, a legend that names the gains, no title. `fig.savefig('data/figure_<label>.png', dpi=150, bbox_inches='tight')` saves it next to the data.
2. Two flights with the same gains: put their heights, attitudes and batteries side by side. What differed between the two?
3. For a flight that overshot: where was the pitch when the position passed the target, and how long after the feedback changed sign did the pitch follow?
